<a href="https://colab.research.google.com/github/phfmafin2025/keep-hf-spaces-awake/blob/main/FRAQX_AI_Avatar_Studio_v1_2_3_RECUPERABLE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FRAQX AI Avatar Studio v1.2.3 — Segmentos recuperables

**Objetivo:** conservar v1.2.1/v1.2.2, no reinstalar checkpoints guardados en Google Drive y registrar cada segmento MP4 terminado para poder reanudar después de un corte de sesión.

**IMPORTANTE**: Todas las autorizaciones están en `False` inicialmente. **No ejecutar «Ejecutar todas»**. Las celdas 1 y 3 son de solo lectura; la 2 monta Google Drive únicamente si se autoriza. La 4 publica código de v1.2.3 en Drive solo con autorización; la 5 recupera dependencias temporales únicamente con GPU y autorización; la 6 abre Gradio únicamente con GPU y autorización. **No se garantiza que el video conserve gestos continuos en cada unión.**

**Rutas:** Archivos originales en `/content/drive/MyDrive/FRAQX_AI_Avatar_Studio`, resultados nuevos solo en `outputs/v1_2_3`. El repositorio y el Python temporales en `/content` pueden requerir reconstrucción tras reiniciar Colab. La GPU Colab gratuita puede no estar disponible.

**Garantía de alcance:** Protege los archivos originales; valida integridad SHA-256 y ffprobe del output generado. No hay pruebas reales con T4 en esta versión: las pruebas realizadas fueron de software/mux con videos sintéticos y ejecución de SadTalker simulada.


In [1]:
from google.colab import userdata

token = userdata.get("HF_TOKEN")

if token and token.startswith("hf_"):
    print("FRAQX: TOKEN DISPONIBLE")
else:
    print("FRAQX: TOKEN NO DISPONIBLE")

FRAQX: TOKEN DISPONIBLE


In [2]:

from google.colab import userdata
from huggingface_hub import HfApi

print("=== FRAQX: CONEXION HUGGING FACE ===")

token = userdata.get("HF_TOKEN")
assert token and token.startswith("hf_"), "HF_TOKEN no disponible"

api = HfApi(token=token)
cuenta = api.whoami()

usuario = cuenta["name"]
repo_id = f"{usuario}/FRAQX_AI_AVATAR_STUDIO"

print("AUTENTICACION: CORRECTA")
print("USUARIO:", usuario)
print("REPOSITORIO:", repo_id)

repo = api.repo_info(
    repo_id=repo_id,
    repo_type="dataset"
)

print("ACCESO AL REPOSITORIO: CORRECTO")
print("PRIVADO:", repo.private)
print("TIPO: dataset")

assert repo.private, "El repositorio no es privado"

print("FRAQX: CONEXION VALIDADA")


=== FRAQX: CONEXION HUGGING FACE ===
AUTENTICACION: CORRECTA
USUARIO: FRAQX1
REPOSITORIO: FRAQX1/FRAQX_AI_AVATAR_STUDIO
ACCESO AL REPOSITORIO: CORRECTO
PRIVADO: True
TIPO: dataset
FRAQX: CONEXION VALIDADA


In [3]:

from pathlib import Path
from uuid import uuid4
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download

print("=== FRAQX: PRUEBA DE PERSISTENCIA ===")

REPO_ID = "FRAQX1/FRAQX_AI_AVATAR_STUDIO"
TOKEN = userdata.get("HF_TOKEN")
assert TOKEN, "HF_TOKEN no disponible"

api = HfApi(token=TOKEN)

archivo = f"diagnosticos/FRAQX_TEST_{uuid4().hex}.txt"
contenido = b"FRAQX v1.2.3 - Verificacion de almacenamiento"

# 1. Guardar archivo pequeno en el repositorio privado
api.upload_file(
    path_or_fileobj=contenido,
    path_in_repo=archivo,
    repo_id=REPO_ID,
    repo_type="dataset",
    commit_message="FRAQX: verificacion de persistencia"
)
print("ESCRITURA: CORRECTA")

# 2. Recuperar el archivo desde Hugging Face
ruta = hf_hub_download(
    repo_id=REPO_ID,
    filename=archivo,
    repo_type="dataset",
    token=TOKEN,
    force_download=True
)

# 3. Comparar contenido
assert Path(ruta).read_bytes() == contenido, (
    "El archivo recuperado no coincide"
)

print("LECTURA: CORRECTA")
print("INTEGRIDAD: CORRECTA")
print("ARCHIVO:", archivo)
print("FRAQX: PERSISTENCIA VALIDADA")


=== FRAQX: PRUEBA DE PERSISTENCIA ===
ESCRITURA: CORRECTA


(…)EST_7ae9ce894a9945cf9835689bf2d976e9.txt:   0%|          | 0.00/45.0 [00:00<?, ?B/s]

LECTURA: CORRECTA
INTEGRIDAD: CORRECTA
ARCHIVO: diagnosticos/FRAQX_TEST_7ae9ce894a9945cf9835689bf2d976e9.txt
FRAQX: PERSISTENCIA VALIDADA


In [ ]:
# CELDA 1 · Diagnóstico local de solo lectura: sin montar Drive ni instalar nada.
from pathlib import Path
import os, shutil, subprocess, sys
print('=== FRAQX v1.2.2: DIAGNOSTICO LOCAL (LECTURA) ===')
print('Python Colab:',sys.version.split()[0])
print('Drive carpeta:',Path('/content/drive/MyDrive/FRAQX_AI_Avatar_Studio').is_dir())
print('Drive montaje:',os.path.ismount('/content/drive'))
print('SadTalker repo:',Path('/content/SadTalker/inference.py').is_file())
print('Python SadTalker:',Path('/content/miniforge3/envs/sadtalker/bin/python').is_file())
exe=shutil.which('nvidia-smi')
if not exe:
    print('GPU_NVIDIA: NO ASIGNADA (nvidia-smi ausente)')
else:
    try:
        result=subprocess.run([exe,'-L'],text=True,capture_output=True,timeout=15)
        devices=[x for x in result.stdout.splitlines() if x.strip().startswith('GPU ')]
        print('GPU_NVIDIA:',devices if result.returncode==0 and devices else 'NO CONFIRMADA')
        if result.returncode: print('Detalle GPU:',result.stderr.strip()[:300])
    except Exception as error:
        print('GPU_NVIDIA: ERROR DE DIAGNOSTICO:',type(error).__name__,str(error)[:300])
print('FIN_DIAGNOSTICO: no se modificaron archivos ni configuraciones.')


=== FRAQX v1.2.2: DIAGNOSTICO LOCAL (LECTURA) ===
Python Colab: 3.13.16
Drive carpeta: False
Drive montaje: False
SadTalker repo: False
Python SadTalker: False
GPU_NVIDIA: ['GPU 0: Tesla T4 (UUID: GPU-965fb96e-aeb9-1716-1bac-c4c126576c51)']
FIN_DIAGNOSTICO: no se modificaron archivos ni configuraciones.


In [ ]:
# CELDA 2 · Montaje OPCIONAL de Drive. No descarga ni instala modelos.
# Ajusta a True SOLO si autorizas montar la cuenta actual de Google Drive.
AUTORIZO_MONTAR_DRIVE = True
from pathlib import Path
if not AUTORIZO_MONTAR_DRIVE:
    print('MONTAJE_NO_AUTORIZADO: ningún cambio.')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    carpeta = Path('/content/drive/MyDrive/FRAQX_AI_Avatar_Studio')
    print('CARPETA_FRAQX:',carpeta, 'EXISTE:',carpeta.is_dir())
    if not carpeta.is_dir():
        print('DETENER: verifica la cuenta de Drive; no se creará una carpeta nueva.')


MessageError: Error: credential propagation was unsuccessful

In [ ]:
from google.colab import auth

print("=== FRAQX: VERIFICACION DE AUTENTICACION API ===")

try:
    auth.authenticate_user()
    print("AUTH_API: CORRECTA")
except Exception as e:
    print("AUTH_API: FALLIDA")
    print("TIPO:", type(e).__name__)
    print("DETALLE:", str(e)[:300])

print("FIN_VERIFICACION")

=== FRAQX: VERIFICACION DE AUTENTICACION API ===
AUTH_API: FALLIDA
TIPO: MessageError
DETALLE: Error: credential propagation was unsuccessful
FIN_VERIFICACION


In [ ]:
# CELDA 3 · Inventario de solo lectura de Drive y el entorno local.
# No cambia rutas, no guarda registros y no descarga componentes.
from pathlib import Path
import hashlib, os, subprocess
BASE=Path('/content/drive/MyDrive/FRAQX_AI_Avatar_Studio')
REPO=Path('/content/SadTalker')
PY_ST=Path('/content/miniforge3/envs/sadtalker/bin/python')
SADTALKER_MODELS={'SadTalker_V0.0.2_256.safetensors':600_000_000,
 'SadTalker_V0.0.2_512.safetensors':600_000_000,
 'mapping_00109-model.pth.tar':100_000_000,
 'mapping_00229-model.pth.tar':100_000_000}
GFPGAN_MODELS={'GFPGANv1.4.pth':300_000_000,
 'alignment_WFLW_4HG.pth':150_000_000,
 'detection_Resnet50_Final.pth':90_000_000,
 'parsing_parsenet.pth':70_000_000}
print('=== FRAQX: INVENTARIO LECTURA ===')
if not BASE.is_dir():
    print('DRIVE_NO_DISPONIBLE: monta la unidad con la celda 2; no se modificarán archivos.')
else:
    print('BASE:',BASE)
    for name,minimum in SADTALKER_MODELS.items():
        p=BASE/'models'/name
        size=p.stat().st_size if p.is_file() else 0
        print('SADTALKER',name,'TAMANO_BYTES:',size,'TAMANO_MINIMO:',minimum,
              'ESTADO:', 'PRESENTE_MINIMO' if size>=minimum else 'FALTA_O_INCOMPLETO')
    for name,minimum in GFPGAN_MODELS.items():
        candidates=[BASE/'models/gfpgan/weights'/name,BASE/'models'/name,
                    REPO/'checkpoints'/name,REPO/'gfpgan/weights'/name]
        present=[str(p) for p in candidates if p.is_file() and p.stat().st_size>=minimum]
        print('GFPGAN',name,'UBICACIONES_VALIDAS_POR_TAMANO:',present or 'NO ENCONTRADAS')
    app=BASE/'app/FRAQX_AI_Avatar_Studio_v1_2_1.py'
    print('APP_V121:',str(app),'EXISTE:',app.is_file(),
          'SHA256:',hashlib.sha256(app.read_bytes()).hexdigest() if app.is_file() else 'N/A')
    for name in ('outputs/v1_2_0','outputs/v1_2_1','logs'):
        p=BASE/name
        print('PERSISTENTE:',name,'EXISTE:',p.is_dir())
print('REPO:',REPO/'inference.py','EXISTE:',(REPO/'inference.py').is_file())
print('PYTHON_SADTALKER:',PY_ST,'EXISTE:',PY_ST.is_file())
print('CONDA:',Path('/content/miniforge3/bin/conda').is_file())
print('FIN_INVENTARIO. No se ha escrito nada.')


In [ ]:
# CELDA 4 — Publicación protegida del MOTOR v1.2.3 (no requiere GPU)
# Por defecto: solo inspección. No instalará dependencias ni descargará checkpoints.
from pathlib import Path
import hashlib, base64, zlib
AUTORIZO_INSTALAR_MOTOR = False
BASE = Path('/content/drive/MyDrive/FRAQX_AI_Avatar_Studio')
ARCHIVOS_MOTOR = {
  'fraqx_render_recuperable_v123.py': ('8bdd4a7a64600838aeb3ab602a441a58ee5f0abcd8939eec8dd31206eee686e7', 'eNq9Pcty40aSd34FLIejADcEkepH2GzDEbTE7ta4W9KI6vbMyFwEBIAkWiQAAyBbGoUi9h92T3ub4xx82JjbXPUn+yWbmVUFFB4kpbZn+tAigaqsrKysfFVmcWdn59XZ4I9/0gZH2mDl5m6qjfKlH8baqmftW0+1//vP/9ayYLoIotz1wvt/RFoaeMskSN3LeaAlbupqI9c/d+dXQWp1OsPMS8N8CU+Hfzp4+3509GGgBZEWL/NkmWd7q56z7zx9qQVzzbv/hx9OYy2KtTDKcnfuan6QBJEfRF7oZp0ohO+Z56ZTV/NmgXeVxGGUZ5Y2jLSDeO5eAlraMg/n4V+hQRxpr0/fa24WTiPXd63Ozs5OZ5LGC81xJktAKHAcLVwkcZprbhTFuZuHcZR15KN0ClPJAvkd4OXBdT4PL+WTmZvNlK8fsziSnxduPpOf44yPmsAzaC6HPFWapMUo2QzxL74tL5M09oKswCoPFskknBft83ARdDofhmejo5NjzdYYLRHrDI9fHx0PnbPhhyP5hhbVGQ1fvxsenw8PHaK7c9ZjndHg8Hzw9sfhmXNw8u7d0Tm29vxnXvfZC9Z5dTqC7/vPOz8MRkP4hHjrcWYF0SpM48iaBrkuYGMLZmpsj2gV5Xt+Gq6CvXc3h/SXNxocOZyrHM5VzDA6Z8PTk82gsUUFdMFh2P/0z87ofDMAalKBsAijcBKn0+DpHrTP9jLXzwni3mUY7SU3+SyOEPjo4GxwfvDG2T5/0bIyyiR1f7l2Vr39pw7sAzf3ZgTz6C9DJKvO9p+/wPbPe/vM6By9G7weOsM/neO7W2Z9TKb4Ev4G/EMS8b+fgsuE3XUG7w+PTpT2n9wVvV4kT/nfZy79ncxdjz644m88nUL3169OXw+OnZ+GR6/fnHN0+CPY6M+sJJ9hW3cO+wf3uvPTq7c/Oc/evOZvOlrjH/ODPPBwFzlnQRYF+fOu8yqM3LmEhRsqjKYObSx4T8+NTqfjzd0s04iKwzSNU/1sGSFr0xejT2Ml0ASa+sEEdokLdNNxR4mXM8Be7EdLvDXoxacwn/E1o9ZWDPJEZ+klM0AyaJOkX8wDeEG7nMfeFQgfLcyDVJ+7i0vf7UMrKw1cX+91959pX2v4xzC1S8aMfoUKM2uZ+G4e6ASGj58GIGkieDULrkG4BVmuG2IWbh4vQs9BuUHImVp8+VGABFl1/z9R6LlIeSFUFy4wbAgQ4gzkoMYFpwYCIgkD3/XxGbwMtEWcx6mF0o6TLZ9JtiUS0NN8kcBD/G4hhZwIBqLXFn7SngCPQBOLwSfgdRDekR8v9GcGTkM3BIz0ppw/0Rm7cAJfw3KD1I59WG6bLfPJ7jeC4lWS4eQtf7lIdJi7qU1MID6I+9zex/4ZCmk388LQfuXOs8Co9J0AXy+zmV59CuhOspvI0+E1CMooluiKl2mQwG4IdMDV1EqCTJBP58qEcC7LaB5GV/oizIht4yv7PF0Gcv3CxWKZo85zvDi50bN4mXqBiSoqL1fxONay+DINUAlehqm2jECz5MHE9fJYC67DDKVE8FLLQlg8LfTv/w6M78UmMI7QZOVSqiPIJeXPgB3pG41NbcMJNbNoiExXOBXegLLjb8PMQSLphgbML/YNwdC+sOV3MUJ13VI3BF2rbNgJOzg5fvX26OD8xBmcHbw5+oCC6Wh0jsqmr90i1DtWrgTfFwovEj4lL9LXz+RFrkQtXBWanaQbQDFUMogJ4uNHz5e9RRMjCV20VjK0fu7/oY3eDABCX0NUszzlpCyH/FI7TWOQkNxoAg2RBenK9eMUZEy0jDwXqAL6fe7CoqM9hN3DKLZKnlRnKfiZOJSYmS8cjOKBtEplb+CaeYzsFUl+KyAE116Q5NrJiKZUhfyl9uo96DuQKa/jeAp2HelwLYn99P5XmrTrL8IcOHrmpj4ikb3U3MsUHgBY2JfhSkG8kBC0rEJECBkMhqCpSA4pmsMo6Td0TG1lQWbo0M5EGG0r+3mszOgPTh2sy2Cahihc8xRwglXOaMeCOdq6yp8hRT6BcRw4U1CHYB0EvhNHXrBGlpwuL+chcUm2nOfAOMBDC5D/l/DR1D66i/u/ZSULaXGewiql3gyWohQ1v780+d3kxmjw9uhwsFlYbBK6gqJAFFCVvu4tfFObx1P7OI6giffJp08CI0Ac3mlhpuHDEksPhVFpeFvpMtLnMGt94SY6LLcJYA2Dg8PVh78GAoO/sGdhcjRai3VU/vPchNwP7gIRN8AOAOdCMIZCWw9ND5w8qNJgOyljnHisTYAJQRzpt5XudwaQFrC/6I7v8FNiZbkfpOnFbm+/2+2PmxTXeBvAs2ZLAeWkKeWu0/TQq0QYvljE6Tr7ObJtWwNPY3B8CFa/bf8c4U5iaOeCP1fQWSNC4xtoYFQg1fV+6m1ZM+23Lxqng03SitPNVkYcnR+evD83FGq3TH0CMwGdeA5u1iHqxNS72zg1QDD1Shq2LPjp2cnBcHTivBq8fXv/X7DgANKwtA9BiswNQ8D/d0xuDBQVgWo1C1NAMY8rBoHyHLxxMFzhj5OFfw2072ytuwExNhBSx11mKHP2Vq53/2tcikwCyicJBrMLq0eG4Dx2/UyX+/eCTSaEMjPZLjg2LEDg+CWbxZ8cgBO4i6z4DvY7uN34NZ7A/wiQmeVo48JScL2rDA0OGJh7bwUk7WLMG63gfQQbUtcz8gtQHcmeKPF4P1xmz8lvkgA4HniarUI/AIfWJHbikNzPg+Ry11iF5AOkCVAo10vM5Zy1W1h2euKDfYQeGLpbkUsOrLLSGJeANQY1hfzo0yr7W1fzEGBymwXU/ur+b/PQd1vXUkiN2xKLvgYyWNClr61MObO+5sLny5s8yODzWka7k64SdnIkVJWFBfsorF0oLHhzIYYbN+V8i9qfF9oyiineE4JW5mNvmi4fqJjyWOBMk3bCaBIL7y64TsA7Dnyhj9Lgl2WI/g3C55J/25TIbqfROEXHuH56BZAGO0dpJ6ZvbJr3KOC4BGnAkdZuNkx6JbZOgYNEDowlfcWZ8FPoo8PfBfH9ndbb/wbRVF7PgnA6y5X3m7A7C8CAXXL2C6ON3Af2EyAnBnFXU2eSggPh4HPcDt29HmvxFig4By0pcJHJ8Nwr8aBslmTFBpQvdQQtRhfG9JD+wLt+rWdXXUR9/5n1HPcdvvsOw2vWc2OjpMcwXBgFGYZZQcPrt9ATRf0InX6KvboRgNGwXQBbEOkiFfqqFB2rmpTgEqDGwAW/SY7F3YNoI+MSg7mXmQ5Qd4sWhvY9CJdrvWs9fW5q+3uAxub5lDLFl1x3u/L71tPJXYZGZJIGK9DgrnYrhxDvdD41g23HUkw2uuSMkMFs8T0IYJ1vQXa8N1CDOA3/KgLCEeNeKFDG1YADYIDkiIAaabwEvVXg8zWuhoGkedp0ZdrWOM7jKYwBxpMfovOGkR4XNLmkBuzI2+iuTgKF+z6482XQ4s1R7EyV0J0yZqeI6hofSMntKG1WfmkmiX3elyTiX8egteQWL16J7/Cu7A0czDC4lkF7Qdp+y5oVWJCSLJtwnYkhAqZCTcJrZ7LIy3bygSGVTwFIr6oI2AmFGl0DHHWSI7UW782/jXkjqa+mydLBkOGNdJWCaxRNwn39NAu9GcwRphW6u9kiZBUJD237dUOcwl+o0os+0rbSdHH0EdJ5h8GaEg7syqZ9fAHDgK30Fpd5s0Ni0ikDGr+95xVZp4vIganCPudth9cJ6CSfPAFovm5CKMLhdTF/NIFLGxqXBCmSBlYWoG7WU/YfMMmfsyc/+0/6zMTm3DA3oc07Y90wumgIFiTCLLvhN4bHRFzjT0Kwp3yXGRe7T9ElUvcMJ0XZFf6kYYK2SjIP8zkKZ90A36owtoPJHJleR6axKcKPSh92cRrYAi+puoFfVAtg4oJ/j0rjotCuZHYL6UWnB33RynITPCLT2TkIjvv/jQslGVe5Co8v0Lz3wxSYst65EuQBWizA+kE9E3MJJZmtBKlgrjDalYmsD4grG6AeeIyvmqOfvneOT5zDo9HpyfHRD2+HXL8DkMoUdDwF2mNgUoGpEnmBldwwxWVpgD0LkjgLgd5gFhXnRTi7eey5GFqt04gOiTYAPKVDoS2w0MrHbU56BlyYBR3eFL5MMxJbkQr4sTEwgUMXed14FCjF8SasQM750LW61r5zS4a0lbmTIA+iLE6ztsMbBq5yguGpbrfX/XZ3AbuPjm2s3E1ZwYDomuCJFXealT77+/U+zXmK5VNObtkeIr5pCcvGkgU5a1C/khdpU1WPciRJqodc/brqVhCbTpKpG+19IlX1ANw45A14IWyxldEcad0yzZBuq7BGOUncaYDM9loWkMkD4RjE5EuQulkgv1ne0ndxHu7KDecYO9ONlxprgqh0cpwVRjvjyHHQKUvzDMM/+k7P6u1bvR2AkKSo3HcO3h8OnJMfdwypBCuhLhFwQQobD1c0T583bKymZmBiZCblYimbm+ZWYyef4yz3EASX/WHkgZVBaQOwnHGKrq6LvoYv/KC1OgSUxf5zUBZVlMFig3UsYxnTkGITB8ws6cGg0S4dhsKrN8PBIRhIUq20MSo0V9eifm5vbJs2RQgXYa5IMD8kYYr8ewvg75o2ZeHRNFR5C10/EMm4YY+03Xsd5qWzhpr+or+/3x1XHGgOQ57tBjlYvtNMDxfuFFiCrDMwFAJYGT8zZeaJUz4AwYSGRDifF/pVUEK0Kbwf8d3YoFSbDuhavargJ+Pn9EyGz+lNRRjQ62qMjT/KlpNJeG3N408BaGeJUpkOsMlBPkIISMR57qI9iLF+FPQyYMPPadCoUVEh5Kqo8EftqJSZBhvDfkSLR2Kihxll+9C5h1hUHSSLSWsG7hPKzmrgSq4jvZKL/D342MaDwlh9rQu4Rff/XAQp+FZoI4Srmimgk3uOAq7GcAY+f9qloSVXVRsgVenkCAm4JfJCPfl5/nNw9QAuQAMnEh7hpkzjTKDFDzoKTwwYrhYTq3KbJEqFQA0gTzRr//kD3fQsnodeCEYhuOQCInjjC/cGOOeXpQiTwavqENxpF3MIJhNMD1mRK1RDRcEZjIwuNzAWYaSX+7zag4OMhJsIEIXzXYzBve9iTWXD7yhEggaCfPI9kv1r7UWXd3lYJFQEonDherBwvW+65dLJJQumJXatnETj8baJG2W2MPe9ZQpmms0jRmAWgmHOH30ncVbOuvMgsTmd5GimbLXLe6mnOlluF291/voJglAPqYcrWGZMVLj/5xwUcixFbkyT3X32LdhXMlBhwvbxYMuHrbl/ivrqfoeDf7f/dYXCcgZPbHxbTgqpIXWKQNMkNEs8BfI2PpbkRizJbbploc/6E3YbPun1u0/9O4z/kebkk2d9+tJiRslAhCCl7AVPOH/RVwxvmc+Nlu6SO8sOHJDocddU0KgDTA7W5G0NFLhBtFzQibBOxDAEZ0ymmOlVBiSEjcb6msgBBIyDaArOqEMhI3pXSwZU4hmkfRx+YAs04Qe3UomJaEntLZczDRgg6Vm/TZkVcKhFm45R4yvVPS53CgxefWE241cOmteY/+QrvYTsaGldyImydfFIaY8GKbr2MhrVl7unFhSCscgMYWSHsP5lHM91+owEEFaJwz0M8VaaKiqkqoAAgrZIDbNoRzPkn1Qgru/wREbHI2uP9euGogxWiZQFUFRky8tjOMzHyvTbq/6K2PPKXCFHAu9ZoHwXmU5Hp1dFLHXtqpmbyG3cbT5vBXkP++4quMm4b5AFKF2AYpmtA2ST9ZlhWHTyHOjy3Flukgv2Mb50QAaMy8xXNEJrKYLK3A01Qe+i3/umEvsBkMI8RbiTeA5egM6HqB6lpoE1Wc7nC0z01FMx8oW7O+nufju+7X0DgqjSbY1Jd4iiNseg/sem4SlwwoDOHhNp1GyP8XxAtsfhC3QxBuNQ4iBOjCNe5FNSYiB/tsdkI4sOTtU5Jdz2km7wBrzPBcYYi6zAK10AgQKfySKon/3SWIQ0eoV6I7Gg0OfYlYdqxUpT6gmHzvPGgHPLRnWZSK1rcnHTzN4VuZewHMDAU2BJbuKiLworpZEkBsugukiIgUwXBNM1xNBN4ADKOrCUWfEhZOyPpoBUKTmtwtGCdIK8zoOWUdBlbTJgBVrbWnPL4MC9/xVsHph5GvDsNzzOBrPQnYORc6MFRaq/H8yVfNV+USGQUjpaDWwWau7iEpNDiahKnpV2/yulYIk02Bv0ImbuTWGRQFOeEwVWVzXbLE7DKSZjZQptuJyqBj6IJMBh0IwyRvSisZSxhpVO57Bg7GtrkTwDFkSitTWTwVVuuwpbTiG0HOoBZ0HFTgKZh9SlsMS702dAgWJvEWVflke3yEQgHjE7D94v3TmrpcvChuAH0VnqkeDWCyLt6VyNR+wJ8Vqp08eGKeyBzdJaqwDjGl8+EUBLMwCAcpZviVgAoYgJJa9yF0ZJ+FZTzFKvBUI7RV8tZSY1ERQYSfIW0hDFApr1nHIgrp7UDpdbojEVNNvRoCYwJAxtLa6QO/gXodOoc5kQ2Aahnu+WemaZEiD/ye1LVQBCTvQrQsMEw4TIXxiM0l5cu6zb7UhGJyZTGmoDnCQFZyHIwFq5vTMZpUjyj5jrsYTH7OBsODgfHrKqeazmx1cElCm/lVQQmqRV3anCrtQIAndiJ8B/W07osRIjnLv8rGiqnGHPC3VdZFqqmV0PkEZlJuWDtqVE/KJqxI+3TUSEiwoRSwUDPogTzy0iM+vR2bqxS7Qq3sNWtHjoqBWruJmXyGloknYtlebGLaYUEOi1JTBEx/rbUrSr7ysR+9r25Pq8Bn7jKq6DxMOeTUjbF6DzL5MKv4dE+Cxp8BhJUOMQ8X5Lis6XGuzvyzhFA0XkgGU8mhFgMCTVYg8zrql8A6DhyahQJFbVaJSHvwUfF0uV6QrTCnUBVLLx64WkmzjuXUztR/AQzzOkLLLP4JdqfHraWhICz40v7I3CZq2gkUNX9rbMC21InvXR6Ur4QSCzTsSsEy8Pw6XmZ8HkubVSuFRenGKQgaSDuqrgo05D3wT+ACFA7InZf9WsvluGU2L9otYKGL8aYCH1buIpd4wBgElssk+zIHKWucf6GI/AE6IJftDZV3/e/Wqx+5V//tWb/lfv+l+N/sJMajJdUANDRJxI7WVBDvjjWYte7kITU16UN3wK8PCCpjEW2X6d+jZs9zc4cxeEWmYk0NaQihNKkgdDkw3JuwefLjCWNwZWTt1PZIGTasKuto3P5GE0D65wG12sHy6UXar86qTxUQG+fEI4deqFUq2eEx+gv9ldXICccKlQYx5wwYGHDnRGIkWNNDdr5UJqCkuHS6lRqM2Wl2CKNBwxzNn08iWlBpQ2iVwdcDH9OMlL6Sbi16JEB7nMVvJVERGzSFclIimBzbFy8I4EUC10msIXNn9xwR6yNXnRFPrRRZwZiENpji+puCpQKuiCFloVoRdaDYV8m/apWHa+V2m+tFeL3vxwm50ND96fDs8GhyfO4XB0OHTACeszs6hOMyWvxkyBSkn8ilGvZBCV2UFopvCc3QpuWFqxUTv46kYRcl5Iwt6LK6oBLlKgWznXt2tHR35WrZ7CdEZ/lyRsPf45pszO77tWb//xFp1f2ss8vdFtWeCmsecL9e2YfLr2Vu1aVqvIennrPMD8CDe9OQyBMcBPvtExQyu8thkv0eY0gaUNU8pYUMu+DUpjE/XbleIum5dN8TdG6zrQUiolDSIfaHcG+825dKMooJqGOSZgrYK5WucQxVnuhxF+vGnL+dgNmaAJtMiZuX7J2jqvCDBWhLMefcIxey+63S5+8/ou/Em8hZP1XoDGMmG2Y+ROGzm0zi41jsKSyt/MQLwMjwc/wEDFWkNRFlFWHbcVz2ExZMHSCg9Vd54Mq1d2IGqlT7N4HijbEJ4hT2zQTUqzNhdEa3HzvdkyurJ5n3W7l9q0a54WklNrY7cprL9/8F5VRXDhxVbinS0VYjjs5+w3b5n/5t3GKfRv2WloNRBt5XkgOE+4/Ti30O5rkL4Nzr9uu33eyo8qF7nwQ/yyyuchW40zXqfBE9IIjGDbOMXlGjKtp7L9TLnatWyechPykteszgDlWRdTm1X8dW/h2xeUO2dWU0eB6rvi6IqQYjzSiY/p4pKII0cLq+DaUVZzlw+IcV9misEJLB4JmjQX+IYihyfzwZheGidl0QxNtI84FilU0JvOEAX2SjGr4AYTk+oogYy3mIdJZl8kPMCLgV1JhWrcmiKrik+FVltSq7f6vltgNg9ApiBoMOd6G3NEymQymKW//Bhrt2XnOwxbv0QdL8tUQLSApSoi+ssotrS3aqmiykcIgWdUf6n9FKc4ROHC0WVEPbNI0kO//P5veXFlxo0WfAxAyJDJjf6jkGxWce3JydmPwzMMGjLGtl7RUyZkAnfewH903c+W2iHxndgK/EdqnaWehfm8mTVxgfOCaIaZT6nsK7/LnRanDl2KABTt8LmbJGRNUd8vs4Gw0vWUR/dvMstNp6uLXv8ZmOpo1rYklK6vrQzmIK+V6spVvyskoii0RK8DfBtmisJJu15qVanAROnOETeEQKScXq4a2/JBlUJoBGcrp4M4lyLpsijbHAOLXNTrvcadCeyK1nIt8lywVqw4wO7uoQym+jDuTO4/t7odbrHYG1ZEVyaHlucs9m3hhzLzcuosk8xFvy/lhecdUpGCG9DbdEiopgQGF9UwCekkM6n+xGbAjNf7L54x8xf07PIb+xtz4YIEceiGGdqydu8F6UsOql+khBP2OD2RMSQaCDHjoFPPc1OMTgtvPEJ13lT0Z6c0TItJFV8lqdgusCtSvE/MJb/2+q7Ca14f+RKDoaV+pPuL2O4lfel9u38lqn/THDZD0XMRryZzd4oc+mTigvWEKptjoGwaYMeSE42OcPiEcHh1dDx4e/QX8PtYxZtDUSEVm5rTAaz1SdFiivXIH1UVl7z2xBYPZO65w33wLBSqhwR34fTVG3MxX7YpTB+h7DBPvCLrTERSWMfjNdqkYS9j5ALohXEcgR3fdhSnymT2A2xMSrxSi16NUiNRW65aKKOjKBEtakGpGsysVHqZZemWQG1i00AXQksVJ5ok42CYv4YJRwtk39jkbeFTzXaObkiQXFyNMVAxgb88zQUhIHbGA20mcQ0T1S7OAkzWnN7/PQrirOX4UMYSEaUikyQUCy5oaWqYOiItIoqMO9XKYZWxdnZ2PtDQqNzAQvdAokXcjDO1hEonqYQ5WlKSIrfs+AE7FogGczpPplGKK0G2rHV5xQnhxXm6/P5ZbjdR4VF+wPodhf/wEN8tt4rkJySPlV/nTAn5vwXoKRgImQZigZc+GdxQoUAERvpBP9GFGkAYMMmwLpOu5Vksr9X0RhqT3/HA01aYuMpiB2mgsR3iBl7kjldEZPF8hbHZ4i6qHbZjajvs558Z2zGewIdop7Z3DLORClMMT3OsT7lFhvzr3COQuROS1UhmEsnuBO0Aku0g+4lArd1qeqAq81GL0mxaHKFVPWZJH81qUX3t0jDhPFHXi3pp63i3qGX+HiuZqZDZ6n7zNep5vgbb77I5HI5eAec6H44Ohyf9clv6WN1Mw+5Uh90Zi7JmWeFbr3dW1rmiMPi2gwH+rascigXhy6qIqwdo9Bqoz1XvdQZq1/VrvOcaz9QvBEPu4Myxni2ePTcfxxUFU5BN0VeioEU9vysURn1s4wHud13+Nu82+8Iu0x1l060qjgflqUelPqpxD1YzJISzEDrOwYKNbOaEE0fEsdRQ91UY+cUxg5KVKGNt8l5N9qQaBr/gKbfjJ8wZHR073MYpjoYQqjgZajIMPyraCvbgpAJWnDOAVlNOk/jRdXFwhMMW5xHr0ujqDNhYtkdyYGMCMjg53s6WhfpXYn2YZtaM0lezQpAMjaSsL2x83Hbg8/DBywPYzccC8rAaBqwOw6nB12V8gQsytj/jvHV9FtLGQ8/aLkCwHYVkD8wu5UShQ0O81E05LOQW23rbEl1Mue3AjaMk0wenHPASPLtWtE4RfL4pTFm5Ts9qCe3liSCH03Ykeno2PHgzPPjR+eHtyR/fD8Gzwqmwl/LaL97TqFQ74dG7/Mw1+sOOflQr8oF5d6gr5PKKTC4LnjH58nFZfMKZPD8b/DD4w0lfnMyUaTfiFuaTEbyiWBkRuty8jRNEVLX25kPDTrkFRCrN2GYgHk/PTl6fDUcj1nk0O1fvyeFBNFEKoKJbq+d52HlJ9aB1UpAEq8umFzuhvzO+27uVxNmRYHbAm8OCHk3n7cqAN5hS07vMYA3alfvuk+PDJrG3JiaQ8qiX88IaU28dhU/7LsA8le3ARfNGha/ETxSLi/GaiaWCuTTtbPj+/IhHKUgRng2PD4dn6wnATUPyCYPKC34atfVkDJnQrBhUjYOvyl7A+3XLfdJe0Szm3JzlI/1Icc7wKE+yzhzieLlxZKFII6KUKeM8qnzknylyP24SaZ0BUA67Nd+iwQZcFBaHiDI9prVxq/moDC6AtY/0oEwK2jMSTi2Zop17Bz85r98Pzg6RfT8Mz45eHR004m01lmnddsWVZxv2DCC3nVCPZDkRhvt8luPz8O2WUKLksIczkgD2m9lISaN6BCfJ0X8HPpLx80ewkohyPpibChqtT1GoqD00Jemk4IJzOStYby1P8hoQaj6umeJk4tmPcY3q24Abicp1Z2uVBPmcjqIowOCg3o9XEuTt2hcter01Q09xw9Yl6LWUwmYt5kXVXpigEW+XrqEig7f6g4RI29BroLW6gdVlFOQgrDqNDdqM8ZIO5StQRHUbm3yTIzNpdWKo7e/pxTR46ANdzryFgerm58HJu9O3w/Mh6/xWd+r2ql+WD+6tLjg1xoZaoVohm6xVves86C6TpuF8Pjw7e3+Kiei1RnMX1AwPmo3tygVmnzdBJW9citX2y0/sbuP6E7vX5TkT8oIxskCE9VW5bEwIoS81fkVLSj+Oc1PcdTOA+Y7woIBfHu+5aQJYZnuUXJrKyjxrg7NI2Q5ySFnc/Dv7hei91i6IEYUSlftdzNa6aUKQZ5ZU0FPkrt1SpKkMY9RqghWXrtXlLo8Kl4ugWi9MFQfltat4IpBqLh7Th+JSmGhPXMIqf9EIzx9E0SU0FnnzckloVLtZnVx4hfb62igxId5j40zCKEOZRa8r0/lN4xMtON9FoSZ/KsOkUkf+a08+RYeCeYwNlhmlqgH/Wqqz/YeTH/qyuho87PPB+Xt0r6sbu7wyTbiy6yNo6h0bsBzZwxPJ60nE4Nqq2cEMLF9ATFwRgKA5VHI6QYZzjdPWQnqORiXKwOPKgMtVfyVDk3O8SHqraFxXqrxwYeNhDol6Yb6b2PI3saxBOl0imU7xG52qw5qRYLW5KlV/K4z7U+qPhMkLUpaXme0mluuDUbi8JNBpRmf0NuMpG6y4I9FXVI2f3uDl6xn15N1wUYQ4YuYsmCc2G+EPX2AyC9jAJv38BU8I4j+QE8ijFQBGcFwxJ73I3/JmcejBwtPVUKaokLDFbzWt6yrNV46/zbhdmAPuohPs7CbyUvBL3M+IZvjrLyb/AQ/5c2Y3jdvJSrANXERaWwsF21qLbLcHti4vGsdry3mmTUGj7vp+YoPtFpdEUHe870l27m3q/bCFae1KvvnGdXn8YgIbNReTy/tyKd1oiQp3mfEfZ6gXuecFK0LHBgog0HZBbLQtC0rj5uhCRsvhD+Io47VronANpWrxc0QgToGrWAnvMeNDM9q/NDJ2ykhoFEofn1iCdLayP5WYYt2SoC6kqOkTp39DoEqboY/X+uAFgNXLFrm7UdgTit1OiVGUFCXsEelC7oJV27A5u21w99tnJ6neV1UkNRGK8KUE2t6/EAAlcaCjiZI8swuzkPoIg4S6k/FDH4urcTgJa7fkbSrer9GcW0gN8uPk+yVKwqZRZ6jqJGH4c3+h6FXRN91OB6jg0CXLjkNXizoOah7HEUTgxuLoBjbJYngd4s95gF4C5ff/jHbiEw=='),
  'FRAQX_AI_Avatar_Studio_v1_2_3.py': ('414a7dd1d8716a025ebc31ec9815ca7bc528cae0876dd419062e2e110ccf29e9', 'eNq9V91u2zYUvs9TEOkFpVRV4rTdRQdduImTGVjTzG6HDVmg0RLtsJFIlT9O0yLAHmKvsatdDNjl+iZ7kp1DSracpkG3iwkwTFHk+fn4nR9ub2+PpeV6zt6T5SDdTx+Tv3/5lTSaG66XLMwNErJwTJeMWFUqwiVRzjbOmt3lIN/PH6fb29tbc61q0jB7UYkZEXWjtCWn8LrVjpXpRgvNSqEIMzAK2+aavX2Xay5LruGvcA3XbFbxfDnYf9wJW3AJs5YnBIxzNfwLaRpe2PyNmiXkaDL87oeR1kon5PlwOtra2ir5fLUrdyISNVvANuZAfUJKB/NCyYQYvqi5tDAQ7+G7saKqwOV5s2AyfrZF4LH6OgzwEXMilSVeHFHav3ih6yX4aCYM79kV0SmveFGATkYc/ObKKgBj/vE3Rq5hJghJabwSA57lokzmouIm61xp/QhuzCvFbNT5gtbsxYmQNmqdihNjdYSOxclMqSry3rXj1sW1Ps2t05JEXmG64Daiml3ROOlNhE0wt+EsPnN6PDoZTYaHL8mP5PvRZHw0PoCXZ+RD8OPmJ0nJQ0LhL32jhGzVLFnluIniOAnLgjn8XcEbS0b+D10DvsDcGuIA70KnAdw59UiDMnvd8AiWxmmeS1bzPL+BWZi4oTHxfINxS49AJSRHq/uO4w5fsvCXApyiidaQ5e3pBElR34Ueol8E6JxORsOT14e3MXv4WcT+R6S6YLsXqppblvXCMtoEbQ1LoeRcLDJcf0bDCz3vfa2bilueAaSRZyouDGg1GmKGG0OTDzexnzJnVJT0fPUeEAYHIBgM2N0qO6NtSBh63j8fY5Xm5X9U1Z3dF2sLbNgeGctKBWAHBODNOlgJpz0N25QhU1a+YtUl17CsQ+Rm90PFZXSHjo2tx0enx8MT2Ne5d+++IyEZ0AqWV8LYKJg0x0n8vv2fSIZZB5m1waMrYS9wyfNKFZcmssJWPAtsJMMxGS6ZZZpMLWa2thwBEUFTyes2ucLuF0xflupKRvQBuXcvRE1rHt3ZmfjqIt4D7KRXYkiD59YBl+7skAMlQ+2rAMhalRz/r8lSlFyhJaQCg5ZcG0CBG8KwfArA2KRkpe6AQbHspBIIV1Gi3reOw3yopaGQHiu1ACMOtVjy3v5RRV6cPiH+DNBaMeO+Ynz8U4qCkQaOiaEpvgYg46yQjvUEfAs2OhksBBs0X/tIGrACVPsaLxG0gtUzAR+WwgiAxDtZq6WoBW7o6lF3eK/YLKInji8VxD6bsTeKxmsadKsm6iqKN+uhL1sZfBvjIMLkk1FMaNg10MQopwtIpGfUNVDUMMwqNuMVEGRdJ3u1ER/vP4oc4uBfiPTr0c+let+TeY/1hgPMpUFlJ66ecR35LJztJQ0eELIBxrWQonY1jIzlTTbo9B1CgS7Ex98lHjochZMlAB7tkaxLdiq+5Vt7YKhwCvwBhU+Txyu5Qflgr1Mw7WT2+XxbInQBKG6C/Vd0RveffkUT+nSwD7gEef6lEznhRlUuWL3KRV8IFjYZqOrggheXM/WuBeuVdrwT/2LFMGB4CXogIqJHj0J/csvyNo3dIfGIVYb3LLYMkNYYEm0STD6HiHYS5T131irIJce+u9JkdxVOmtTC1GuWA0ZaQLhnUBiAy/q6JwwaYuwRMDz4O4vmtSaND0mDuQKODRIFMK5ayxOYOlhhIfaDH5vyoIyhwO8x8XTiMCtMxyetb7cMWNWyu3YdvLxjV6g7d5gd6hNQQUAGyZ5soJYWlSguo15nnZz1W9I2UpKurfZddWiqWwvPP+0dCTlrfU76riTBwiTge56A5MJpuCkU13kF9LHZ4I7sNOFMutIfZmhGfM+t+3lqo468BuOgMhA4HDivnR1fVfI0xXKg+QIOT2O+RiJVrL1S+Lj4Gnp/WPHWCa4xl1RQFwhkfjcT2tcPposLsYTC0m/qP0eUPjm+KMxmVuZtt9Wn8rjnM4kgiBWBewd0Hux2XKGA0Lj296/g69UMqJ4Yq9zcGwaa41Xts3zChNvVXo9fR7CnaxGQTZeCXw0+ITEEt6u8FCPkp1Ru9+3fs2+dFHr7ehi2xF53ugkcVRJ82twQMGvXr+4QfvlZ50DSWdRKuIe8G2Rc2QsN2c/0ITZSeJ/dpZuXbho/pD+Tv/5Y1fz1Q4dYYXS4ZB6fvobuBZiFB6mBrqtUDniAMQXkX+xhfNUP1NbI1i245Xa3ApJlhOZ5zYTMcxo4+ICcQjjAzcBLOHkJbcUM4ADWOCxyFSugt/r4J875cz/2t/7U78V2LoVmyPEoTivmZHERmQsGCR4ihcul0EqG/jpE4vSb4WSUH8PNcvwSKtYejbOMDmhS8plbtCXAXKirnGPvGarMHRmm7RmqSl3xMscWAVqDT/Clu7RDGDr3fwCQe4Mg'),
}
print('=== FRAQX v1.2.3: PUBLICACION DE MOTOR ===')
if not BASE.is_dir():
    print('PARADA_SEGURA: Google Drive no está montado o falta la carpeta FRAQX.')
else:
    destino = BASE/'app'
    conflictos = []
    for name, (digest, payload) in ARCHIVOS_MOTOR.items():
        data = zlib.decompress(base64.b64decode(payload))
        actual = hashlib.sha256(data).hexdigest()
        if actual != digest:
            raise ValueError('Código incrustado no supera SHA-256: ' + name)
        path = destino/name
        if path.exists():
            if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != digest:
                conflictos.append(str(path))
            else:
                print('PRESENTE_IDENTICO:', name)
        else:
            print('PENDIENTE_DE_PUBLICAR:', name)
    if conflictos:
        print('PARADA_SEGURA: existen archivos con nombres iguales y contenido distinto.')
        for nombre in conflictos: print('CONFLICTO:', nombre)
    elif not AUTORIZO_INSTALAR_MOTOR:
        print('MODO_LECTURA: ningún archivo publicado. Cambia AUTORIZO_INSTALAR_MOTOR=True cuando decidas.')
    else:
        destino.mkdir(parents=True, exist_ok=True)
        for name, (digest, payload) in ARCHIVOS_MOTOR.items():
            path = destino/name
            if not path.exists():
                data = zlib.decompress(base64.b64decode(payload))
                with path.open('xb') as out:
                    out.write(data)
            if hashlib.sha256(path.read_bytes()).hexdigest() != digest:
                raise RuntimeError('Verificación SHA-256 tras publicación fallida: ' + str(path))
            print('PUBLICADO_VERIFICADO:', str(path))
    print('FIN_PUBLICACION_CONTROLADA')


=== FRAQX v1.2.3: PUBLICACION DE MOTOR ===
PARADA_SEGURA: Google Drive no está montado o falta la carpeta FRAQX.


In [ ]:
# CELDA 5 — Recuperación CON GPU, separada de Gradio (FRAQX v1.2.3)
# Una sola ejecución: monta Drive, recupera repo/pesos/entorno y abre Gradio v1.2.1 con duración configurable.
# Conserva v1.2.0 y sus vídeos; la nueva versión escribe en outputs/v1_2_1.
import base64
import hashlib
import os
from pathlib import Path
import runpy
import shutil
import subprocess
import sys
import time
import traceback

BASE = Path('/content/drive/MyDrive/FRAQX_AI_Avatar_Studio')
REPO = Path('/content/SadTalker')
MFG = Path('/content/miniforge3')
PY_ST = MFG / 'envs/sadtalker/bin/python'
APP_NAME = 'FRAQX_AI_Avatar_Studio_v1_2_1.py'
APP_SHA256 = '0d97a702c2f98963227325d4569afc46ec9f778202409f728a9d6307498087d8'
APP_GZIP_B64 = 'H4sIAAAAAAAC/608XW/cSHLv8ys6XizI0dHUjCwrd+OdC7SypFXOlnSSz3sXrUD0zDRHtPm1/BhJFhTkRwTI8z4eDvcQ7FvyEGD1T/JLUlXdTTY/RpZzZ+x6SHZ3dXVVdX11tb9iB2e7v/8j2z1iuyte8IydF+UiSNhq7G6548FXbPfhP/lCsEWZ8Xnw8HPM5knsB0t4nYWCiZjlYlnGiyRnt2whCjGXvXhZJNHDT0Uw5/A5hHeA6gK8vSTORbbicoYRjMrLnGUiL8OCI5g8iFmULAIfhmZhkuOgM/FjGYgM5gthyiLJ4oSd88U7Hn4UGbsFYDwMFjAcETpMkiWgtpeEfOYO/CyJWMqLqzCYsSBKk6xgp/AqGxa8EEUQCd2i32WrD2suAkBYNx+oDwP1/iGvnyMEqp7zq7IIwuqtnKVZMhd5Xn25rR5pNvVclsFiMPh293yfTQlJ29oEchew5M1FFqzE5tvb1/RLXPN2jzzJNU9yzRoOzvZPT7qDK1pBj9M/vfvu5Ng7f9ftFgVx4CfZUrzYFPEq38z5oqBhm7Mg3kxvi6skBgjf7x8dfvfu3Ht9dAYwaMZNZi39dMljCx+vRbC8KnJrMCiy28mAwR+1wmXGUbp4Dk8DcTMXacGOqGk/y5JM9q3p5c6vxPyjN+dhaF8A0VxxI+ZlgaLnMOt5ZMHfaZDiTxDngGuIj89/xL/lVL+dbrvb2843O9blcC0ig6O3u4f73v4f353Dgu4sN42XCML9kOpfIR+uxSy17ge7f3h9dGL0j9IXspmv6Dfa5vTL+Zx+k6Uc7ofw4X7w9uT1Po60CSNr6+UOS2/Y//7bv7OTLFgGMQ9Z9vBTChLNbNwOhwenh7vHQ8uR/V+Ot3T/86q1pxG22vrG3WgGi1+JLAd5Fjn0GA5kZ0/xt0ZQfocNu+2mxZUGBltuGUcgN973B2++97a/OzRbSRfgXvHORB6L4uXIO8CVmX1SDrPHSw9/BfRRbcPBYLAQPmoB4Ul1Y+PzUImSbGFxUjBYPhFTtuCfjAe5YO95WAoSKdt6m4BegP456QgYBtoGoCbzYIE7hiQu+CRgucgJq4I/nUrYF6NL0DsAFMlnUf8yF56UdxhFnQERW/beunTUuBdK5PKr5NrL+HW376jqO14zSuSg7cQChtZzUnMmijKLCXPHaHOq2ZwmBEXTNBN+iLvTpCjsNthRUQ6zXFxqGiN5URW5Qe4tgswe1jTW/V2epiJe2JZSuaSbkMAw58NPsNgYVTpLmM9BubO1SsuY0dbqJIh9UPjxXLjprTVEJPwgFI9iIWfpKr3NJixzukoZfsEEp6QHGchZyEmyaptUKUw1CyhTBqqTZ7fEdMv3I6VJfB+hC8uYTqEkbYd7fRXMr2w52OjUh5dv4dRojOclYJJxdifH3Rvi3RQSD/5T4tjYYb2sICWcJkEMKh3efauirPd+5I7cLe8OZ7h3c+7Dro/zJMufxjJfkRTnDk2TLgEq9CPoDWoCN6h69Eaj8eg3z2kYKg0XxIk2rtzIeifLXVuP2dpqj3nKetX4L1rQnRp0X4tbTf56OMqHNJasUuRa/TZ5rnE0be+mGtuL2ecwTEWeqBkn7E4C0tgq3aKHVqoDBNaL0m0b3Sk1lXTcgDWG0c7K2K7wuKhEHS0zmUeBepleSUPlRSZ4lNcfgCrgTNF74uMPulnwCx3l1JdOBZ54NX2XlSDghbgp1OOcp7AE4SVlkZbqI42RCwQ3jwPOCNcNE77IbbkONy8WMER2WgULkUCvGMDa9g0x6wb5hKMvLI32JfLmxl2KwrbmIFxzr7hNYWOTGBIMa+iwYzCyEix5wn8jWC4VpwkWekuEg5y+MgArp1If2ibyrIzR99RG8nSbPG8JI6FnbqhnJRJ3FRALIwK079aE+UDDwpb4K+ZdXtQdLoc1v6zrYAFGfgILLmya7EJ9avS6InlsdlPfGv38NIdOKBjUSZKLr5YeuO6RADNYkNxtvdwcW0Nz4Oy2EDgU5Qm4DugP4ccjTUm97htST6Tw9IrMDfDs2bOTWRGA+8JCboRJwMVQRj7Z/CpYkQci+RGJRcDBRDG1MVwA8ffaS6j+A0F7STJiWnGhXny1r2B9HGacxkmaAaG96wyVRJZPx5M4+Shup+O/765TqMAKpcQ0Nh38ZEFqDxs2CIMq0G4+BCaFsDWAIQp3Be2bKRs9LtzHZB7Tkqx1IbIIrSMrY5Nfq4efyD2sg9Wm5OvplFgsRD7PAi0ZtpSPpli8LcENAmPcEAvpFnOYCMQDAtwlIJJgCFwERRkgUmp2LRSKEPUExlIlZtYegqlGgkhnXM+TmYE4+urgBpc1Oq40gAhWR4PGSsz5U8kIZWaQ/nLnlL4f3IAOvRbgIWqXvA6PusgavAgFWPqK3OyABDbJGV9EQQHOeT5hb09fOOz73ffgHG/vOmx3d89hJ4eH7JYdvNndU+hXQWZLyNZv3ao3SEIJigCkHEOCBXrAi2AFPoKNiidLSrCYldSBvt0ZDdsrAou6S3TXrE3AnuoxE3fLv69TJPa/3qkZJ6Otxf3kTk1Lb0OXWR3zbf0h52wkeboEJZPxmmRsnkRpKArU10ocWYgZFD3dj6VAQRU818KsQu59+kEqQQwI3zps8tt7hmTJlGRYJAy8hxifkjqLYEkqDsMNTn0S6UCopAwMVbEmG7vjCfRDLzf4xHE10uhAPCpeAbDaCzw+YatShBBTcBgQL0QGAzLcOXkSBvNA4pQzXGsVybpVGHty9rt9zFBklmWtzwM9JUlDeZ6npYSCiC9FkMjeeTZ3cZ256/O5AO18xSEKyfRY/e5JzkIM4V0HxZUXClAzCaUCUL9GqYNsLEBrKckG79JG5B1E0oXtv7oYT7YhcIRIJ1ljQB43HiIEFpuu2Goy0iaja1tkv2nX0HYdNrkMbTxMw9FnLQxrMhgOEGrTTcP1aSfN8JUgkL5om/3LATgHlanRbLKxbThEtUbQlcayrdHmCDEfoaMgA4etl+5oQIYRtrj2lA9Oz1HMcVUCOXEHk9zDQD8s8ytCfDggJHDqR/hrG7RxwCWAkHIx1Tk0h82WXpnmHLd3NpVOHo7U0oV+DlhrsIkZAUIZATCAyxT+B9Kiwzi1QNRvtna2Ad6PJQ+D4nb6a6ejYSpVyOdZ4s3CZP6R3KDpeGeI6kFOM6niWVod0kwus1YdsqOKMzz0Bm3qMhysE0cdDD+/AgXgzXgMVKL3ENMJKxE2hfSW/o4T4H8Q18IZKElTRKg/aPLqjrBriMUTku36w3jCTWGfT2hzzJNUTjifUDJP5fKez+Tr+DdbHxsbJAOtLoOWKFn5IV/STvmVz/MCVGJWKKSMjdyzKUDopcQpgfMOjo533xz9y+7rk5aQoU6TrgiQFGL4KOJgq9QvSMA1/AVkJIONM4PcKNeEBEk3uQlwC1xmgC5iEBuIWadWWfjPf20R96FfzWF4cYnLsCF+iKcQidwR4HsISqY/xNawp6e1lyBGYBQt9iv433I/QGRtowYDcqCoEsawJaG1A4NWbJs29zMuMtHTIMIUaQ6/Q6QBqo0pgKVnkKupAeX83euTP7xzFDMOOGiAeq8Me0mw9/CzNnsytwg2Ubm00oziNryv1oQqp93K/uFzzqtvaRpD3B4+/AwYqml75nLZmVgGsGT0QTSLMaqXkqLUkLBJjZCT5xiepVM5T57ySxzKzCC5gjB00J4vYdJ8uszcU/UMDvvEdBRr0OgkrnVdaaEARnno2oXloGIKAMz9h79y8PLQqW3FT9qVaXh9T06Drk16Kc+oThx3XaMmzihY0A6CS6YeHtu51FauVVNJ91hPjX2VTgxi7eGp3fNKbx8NA6IldaoBVHeq3AJ58jUnQPgf8+2pY9O5l5/6vfv6sGT9Cg4MLgaxuAGZpOAjYTIsTTBLrP38CTs9Ptz859PDze/3vz1tJmglyxu4yU9fEHm0cJPe+hOwguBjE2KPTQg9NiHy2ITAYxPDjl4BTMoMPLzPRR/0+lRXvIm33wmdWoGligq0Y27IZRfdTPyI0alYeCsUeEB3hBRv7/8qmUQOkUrztPo0VmO/u03l9nGMrTT83NqsI/Atk0U5FzrW6YtxQBnED/8ViUyqXBXluFZrB/amDloLJkFqE+GbriquMXzTJPYMEcgAW7uJ5rCJZ5rkIEyrNsImni0sUNbBARBL0ThyWU+whhSY5/Eo2lkCOhxiDSY+gHcUJhDA1nkNPW8d+1LQ26JUbbva/X8Lywc729PyDRs/kZLRw19jUDYQv7KxRt21Hp20vdV+xUbu1su18zVcA9/a14KF48F/aQJrxuuvUB+kpQCfzYg4rRbEuw6SNQSXGfF7R6Zdq+VhCN/HQ9NVQ5G019tLlinuYNqlEZiHTrvThtLMyHkhnwF2U6aUo0bQ+txkj6/ekprnQzLDvIoqrXDj5JqSrZmPr7b19Z++jr5eeF9/9/Xbr88t8gM9tHZYDeHiX9vQ/UrcXEx21OFkDPEiHUoCWKqWoPNC/EjlBzAfPcAv9ZcBZqu//CgHrMbeljc2h4Dv1J4AP5m9W6i40Uc8JQVGw77LVTRLJsZLlD85bGHz5CEam8cHyLhA9Sd7CdhXC8FzuztY3j2YzKXV6CYdWg9D73q3WI8WBoFTWx9iWe8yPuMf0PLgBD/EdOKuNwzlwyLlCRuDXlc6oOrJjcyZFMm1YypzxxJVMjHp2cwvcDP/EJ+j+4rt+Isgu8GOzlPX3pSHASDpxEraNpltUVuM8tnnIxlHPGvGyzaNNMLp82dMONd81QajoIx3PlLJiTzkxSCTHFzaZVECxiWJg7nd4618xQ5Lni1UEhwQDThZiUIykokVsANojJE8UCrAZ0ot+gHER2BUKCuVZDwEzQhNh1RN49bekDzDRgps2co3rYk67O9Ha3IM4g3NY1YZbIzc0UtKhs2n1mkmUK2ColGSMN5hH7/7ZESQKhDEsoa/U7ahlZytUg0G0pfmgX6v0Zz0Rasu7D48n72wnutUQWfw8LKzNmPYKiaMZZpiLJ9pKeOd0WhkpjLSeeTl451QNLAHSTIn6E0r4BG5mVew9k6O3++fnR+dHHvke1tqC7W4Nv614ppvnelELjKufeKP1Uk2yiToBMwYfxoa7KQyDqWVDZ0O2yHnC08VXnR7K63ZKxa4+qoGZOi0yk7WMP250jMk0YqEhnQjoZ9TuV4s3f8OldfBlQE94qyh6jVIoEgibKFgF94hwFTpC0pYZUlqNaSP9F2/tOmSFgCKnawv5fv57ut3u29+t39mbjfoGqCZp0KilLKFKUZlNSuyZZjMbGvDjdJtY2TzD8bIRsiHrEpbZ7S4jS7blTP1/M0192ZVaslDHw99adB7b0+316ZQamjXnhQASsTf2PW0DsND05BHswXo00kD6wgnb0LRBQYNWfal4VUlNd750bEnU4FEsn7NWaPk1ICbc6mDgbqGw+hnZtcoSVJhhrFft1XBajRWZnsR+H67cU31S0tH/OOW1uxn6hAJNYTKB0UQQc7resam6OhSWrBJyK1GG36s1kNZEN0Nk+j4TMphaJBepV6lt9Sgei+RetmnStwIhvdyvNXLxI5d7tuDXR31HDV845BrfVZfCgsak4rfdba88aG5LvO0v+9PRyH0MsZHHyfsWWA9u1vGYRB/tKMgp1rQhiO7TvQMMW5i3RjVlkk+y+0GJLNGhT2v9kmjdKVdhdUE+luMPXv419U4vdQEjzaQ9ka6WEZczG8x1YXgWW66xTgtObYue5MsGzqqM0W/Lf5N5UG9BzuMZfa4z27ZklxCfEZPr67IN3gqQp7m5GPKo/G2pwlEVG6ow7aMU3YewzJzzNvfNZC0cgoqPHV4a03AWcHwgs4Qkxm8w9/wjOEDvFAyugUATeJEm0RpyyZsliShTS8tW1tx1us4VzCMUl2fiXi7oXULIekbNPN+xhyScK1IxWEv2ojWqYDPgOrmDHqg6XhD+SwTFbzEPEKyVa3SXZmokIRam3AqLaKqrgyt0urY2Je6d3OzqhMRU5sSiZENPfNGAkPEguPK1U51jInMZnOTrxMA3EjS1euQtNHHYdvD7sYPcnI56uRoD9Jqtxjg1ReHsgmKKHr/GhS8r57spp+rN5JLB+zD3sCdMi54Yr4oozS39RAIwDCWK6ZbGPzmePDO83kQqDxDNyLuOfxCb6Yk56650NPd83O6VPCemCiPmUinrJSGWSQtZ9q31iQJOv2ayYVO85emBCi/9wiUx5IRjwwDV3wFXiOvMn0kz5iL72zPdTjUdznwHFHbomdUHfns8p49/If5WVZDwvdXPQnJqpee89mlmvWVCYOKIJ9dbo7Fjmx++62BVSuQNS2wIf1N4yclREcXg669q++kyMNSwxi3VtpsW7vcKgfb6N677maXvsUPmtZz/crWGu5eU81MG6oA6bi9Le0BJlYQpIzseBTg5QIUJKk7Ju64V3r2eJaCBoSOtcroCqwu823IGYlqf995L7tUoXG/9n52bG7rZ124vfGVIXY9aZ+x9liUMkC85mFZX+DRYc4iyNOQY9qtaZ0oGtbXcCp93Q1tWuO79qp5mNwHSRYFGsg4HfAOe7x50OfA13aC6iTkObAUpuETTxT//5UfrZqOH+KD3aM38lAacxB0EN4p4OgcYB5QBQOeWEkT0SgkNM8rZV2LyEWBlwG8WCg+2nV17RsI+gLKE2F9YQH2huMVVFmIkYP6zYNZKHIX2APuYAZKmSCA4grUdRPKXNKVJaMCW3KPjLk06X9bea3DRhrOur8tKs8g1gCpvsX6qxwc6yIU08dz8JJNCxElkigw+i3PPi6Sa6MexvqKPTmRbx0SV2ipsitsOqBRf1mGLNrd2NgPg6Vg8/Lhp1AWYUpbjFWneNEumAWZu7FhaG3rVJ6oVSWgsoizYmKi6qKpslOwuBSrxCGtlYkAwowABsjLvSbQjQ11oRhEbq7vGAdxwYFNr/BCGq+xk5GOeelYxDITLpUowEAuLzggbhkF7JpJZ3iCZVzcotwfJehBcUD7EX6wyWeYQkDDzGoI8lKIuiAceKVjauFuxKIMS5/S5dMLq0yx4tEMQGUcYExDZ3V6mi2X6TJktko+fQnwVoChYmyY4B0YqVlyU0uTmut1Tzl75X3BRBDQwzRjdDkLQYWXK9EuqaLD5enfXrv+KGeMNVVUOy6jmWjF4xKZEVY4gcRiKIoveB06KiN8hIAvhQU19aEkxguXmfQAuQdqmOfw9ghzD80qAauvtowKkwxMz/BQpYloCJZT3h0FMyCxrm6d9iG37bJf/uf35cOfza2pC67VJlVl5f/U9vgpfjbx2cPSOJQHObE8flQTvU1WgXRVUADn8PETbrBFibdsma3SzDqbv4Zduj7Nm5VFQWffqBHp2VbKKZO4W7j6DK/RTK00gw2Y3VqNKkGwHH1AjlGZaN88JBOSmbpNl1DddvMf/ep1Y2Pv4ecoYX4ZY8k7n4Cq++UvtSv/y38zdeEED9fy+sCjobx++UvtF8MIXUSPcUSFB9J13ejWZW4AEanLJxsbVByPoUkugxQyk/hRFdbl5FrhNkR7ubHhmKBTLGIRH+jSvXFYs7FhntFsbDTQOeWZLOVLQUBmvLqaMH75Su7varfDinZGRsGK6rgzelU9q9KJMgc+FRgx1vdFjCnfcMZTcDn1XR550xuvPzTuQ9RuKcYvoE1SdOHX3dtSszAsGZklBTb+8hdDgIDKUoIQQAqiCN4NCNDGBl6AhhbkeSxVo3QzgE5P0FnoGhKjxLWUXQqqtZp/L4NMYqT8FwCOjrtJ8cqj/AJAeyctQI/gh5sALYiEe4DnNNo0KFnK1G1vTH03zzCYnbCtlzvDPnSfDlfl1zfaufW6AOIpsDIViiCLSMpJBvStafKt1fWEpkFUgPbzQqY5srIgD4aZF+FrQ7hl2llSqO78isfgI9S1DfG0eaXMUSf+0wtj2KWjThuqzw2rfWlIV0uXunMQ/Y+N+XQPxzgylaANh8YxsXZaBtUxTJZj2gvDGFX4GnLtdATUaYiV05WIntMIk8lOk1vmVUWIFMsMI/VbDzdloc24roKrTUUPjXpikC65+la7noQ9fOvS9XFidWnxOeo9gVgYhUCA63mY9vU8umfseREPYs+zJutrm7X/8Uh9s7rScHq2v/fd/t7vGEUeE3a2//7ofPeMtM/a8uYWjDdcOpU+/yQr8xifZUEGniUjW9WMLSlbT/9wENoC/Fcp/hyTSUBHDCwlQKqvxoF7uh5lTHF6VPgltW5lCvGfVLlVipNhAJ/EFHgqqBieuTB5KeyhG3LwEq5s4xCXZ9qPWohZuVROsswyUCmKcYmWVEiIBUILquEGKcO8QLugTR3nNcrWhhWP/w8yFK5I8UkAAA=='
COMMIT = 'cd4c046'

# Solo estos 4 checkpoints se usan; ya están en Drive.
SADTALKER_MODELS = {
    'SadTalker_V0.0.2_256.safetensors': 600_000_000,
    'SadTalker_V0.0.2_512.safetensors': 600_000_000,
    'mapping_00109-model.pth.tar': 100_000_000,
    'mapping_00229-model.pth.tar': 100_000_000,
}
GFPGAN_MODELS = {
    'GFPGANv1.4.pth': ('https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth', 300_000_000),
    'alignment_WFLW_4HG.pth': ('https://github.com/xinntao/facexlib/releases/download/v0.1.0/alignment_WFLW_4HG.pth', 150_000_000),
    'detection_Resnet50_Final.pth': ('https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth', 90_000_000),
    'parsing_parsenet.pth': ('https://github.com/xinntao/facexlib/releases/download/v0.2.2/parsing_parsenet.pth', 70_000_000),
}
PY_REQUIREMENTS = [
    'numpy==1.23.4',
    'numba==0.58.1',
    'llvmlite==0.41.1',
    'scipy==1.10.1',
    'face_alignment==1.3.5',
    'imageio==2.19.3',
    'imageio-ffmpeg==0.4.7',
    'librosa==0.9.2',
    'resampy==0.3.1',
    'pydub==0.25.1',
    'kornia==0.6.8',
    'tqdm==4.65.0',
    'yacs==0.1.8',
    'PyYAML==6.0.1',
    'joblib==1.1.0',
    'scikit-image==0.19.3',
    'basicsr==1.4.2',
    'facexlib==0.3.0',
    'gfpgan==1.3.8',
    'av==12.3.0',
    'safetensors==0.4.5',
]
LOG = None


def say(message):
    line = '[FRAQX] ' + str(message)
    print(line, flush=True)
    if LOG is not None:
        with LOG.open('a', encoding='utf-8') as out:
            out.write(line + '\n')


def cmd(arguments, *, cwd=None, env=None):
    """Registrar en Drive los comandos, con error recuperable y mensaje conciso."""
    assert LOG is not None
    say('Ejecutando: ' + ' '.join(map(str, arguments[:5])) + (' ...' if len(arguments)>5 else ''))
    with LOG.open('a', encoding='utf-8') as output:
        proc = subprocess.run([str(a) for a in arguments], cwd=str(cwd) if cwd else None,
                              env=env, stdout=output, stderr=subprocess.STDOUT)
    if proc.returncode:
        tail = '\n'.join(LOG.read_text(encoding='utf-8', errors='replace').splitlines()[-18:])
        raise RuntimeError('Comando falló (código ' + str(proc.returncode) + ').\n' + tail)


def is_good(path, minimum):
    return path.is_file() and path.stat().st_size >= minimum


def init_drive_logger():
    # Drive debe estar montado ANTES, por autorización expresa en una celda separada.
    if not BASE.is_dir():
        raise RuntimeError('Drive sin montar o carpeta FRAQX ausente. No se creará una carpeta nueva.')
    global LOG
    logs = BASE / 'logs' / 'arranque_automatico'
    logs.mkdir(parents=True, exist_ok=True)
    candidate = logs / ('arranque_v1_2_3_' + time.strftime('%Y%m%d_%H%M%S') + '.log')
    if candidate.exists():
        raise RuntimeError('Ya existe un log con este nombre; no se sobrescribe: ' + str(candidate))
    LOG = candidate
    say('Google Drive montado: PASS. Registro de recuperación v1.2.3: ' + str(LOG))


def ensure_gpu_and_utilities():
    if not shutil.which('nvidia-smi'):
        raise RuntimeError('No hay GPU NVIDIA. En Colab selecciona Entorno de ejecución > Cambiar tipo de entorno > T4.')
    cmd(['nvidia-smi', '-L'])
    for binary in ('git', 'curl', 'ffmpeg', 'ffprobe'):
        if not shutil.which(binary):
            raise RuntimeError(f'No se encontró {binary} en la máquina de Colab.')
    say('GPU y utilidades: PASS')


def ensure_app():
    # Nunca sobrescribe versiones existentes, ni siquiera archivos con sufijo VERIFICADA.
    from gzip import decompress
    content = decompress(base64.b64decode(APP_GZIP_B64))
    if hashlib.sha256(content).hexdigest() != APP_SHA256:
        raise RuntimeError('SHA-256 del script integrado no coincide. Aborto sin escribir.')
    app_folder = BASE / 'app'
    if app_folder.exists() and not app_folder.is_dir():
        raise RuntimeError('La ruta de aplicación no es un directorio: ' + str(app_folder))
    app_folder.mkdir(parents=True, exist_ok=True)
    original = app_folder / APP_NAME
    if original.exists() and hashlib.sha256(original.read_bytes()).hexdigest() != APP_SHA256:
        # Se conserva la personalización del usuario y se publica con nombre inmutable.
        original = app_folder / ('FRAQX_AI_Avatar_Studio_v1_2_1_VERIFICADA_' + APP_SHA256[:12] + '.py')
    if original.exists():
        if not original.is_file() or hashlib.sha256(original.read_bytes()).hexdigest() != APP_SHA256:
            raise RuntimeError('Conflicto de aplicación: archivo diferente, no se sobrescribe: ' + str(original))
    else:
        with original.open('xb') as writer:  # falla si alguien crea el archivo antes
            writer.write(content)
    # Nombre temporal exclusivo y estable por contenido; no reemplaza app de /content.
    launch_copy = Path('/content') / ('FRAQX_LANZADOR_' + APP_SHA256[:12] + '.py')
    if launch_copy.exists():
        if hashlib.sha256(launch_copy.read_bytes()).hexdigest() != APP_SHA256:
            raise RuntimeError('Existe lanzador diferente: ' + str(launch_copy))
    else:
        with launch_copy.open('xb') as writer:
            writer.write(content)
    say('Interfaz v1.2.1 íntegra (no sobrescrita): PASS')
    return launch_copy


def ensure_repo():
    if not (REPO / 'inference.py').is_file():
        if REPO.exists():
            raise RuntimeError('Existe /content/SadTalker incompleto; no lo borraré automáticamente.')
        cmd(['git', 'clone', 'https://github.com/OpenTalker/SadTalker.git', str(REPO)])
        cmd(['git', '-C', str(REPO), 'checkout', '--detach', COMMIT])
    rev = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
    if not rev.startswith(COMMIT):
        raise RuntimeError('Versión SadTalker distinta a la validada: ' + rev)
    say('Código SadTalker (commit fijado): PASS')


def link_checkpoint(src, dest, minimum):
    if not is_good(src, minimum):
        raise RuntimeError('Modelo ausente o incompleto: ' + str(src))
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.is_symlink():
        if dest.is_file() and dest.resolve() == src.resolve():
            return
        # Enlace preexistente distinto: detener para revisión, no borrar sin autorizar.
        raise RuntimeError('Enlace de modelo diferente; revisión manual requerida: ' + str(dest))
    elif dest.exists():
        if not is_good(dest, minimum):
            raise RuntimeError('Peso existente incompatible en /content: ' + str(dest))
        return
    dest.symlink_to(src)
    assert is_good(dest, minimum)


def ensure_models():
    for name, minimum in SADTALKER_MODELS.items():
        src = BASE / 'models' / name
        link_checkpoint(src, REPO / 'checkpoints' / name, minimum)
    say('Cuatro checkpoints desde Drive: PASS (sin descargar)')

    store = BASE / 'models' / 'gfpgan' / 'weights'
    store.mkdir(parents=True, exist_ok=True)
    for name, (url, minimum) in GFPGAN_MODELS.items():
        source = store / name
        if not is_good(source, minimum):
            # Comprobar ubicaciones históricas antes de descargar.
            candidates = [BASE / 'models' / name, REPO / 'checkpoints' / name,
                          REPO / 'gfpgan' / 'weights' / name]
            existing = next((x for x in candidates if is_good(x, minimum)), None)
            if existing is not None:
                shutil.copy2(existing, source)
                say(name + ': recuperado de otro directorio')
            else:
                if source.exists():
                    raise RuntimeError('Modelo persistente incompleto, no se reemplaza: ' + str(source))
                part = store / (name + '.part')
                say('Descargando una sola vez: ' + name)
                cmd(['curl', '--fail', '--location', '--retry', '4', '--retry-delay', '3',
                     '--continue-at', '-', '--output', str(part), url])
                if not is_good(part, minimum):
                    raise RuntimeError('Descarga incompleta: ' + str(part))
                part.replace(source)
        link_checkpoint(source, REPO / 'gfpgan' / 'weights' / name, minimum)
        say('GFPGAN ' + name + ': PASS')


def env_ready():
    if not PY_ST.is_file():
        return False
    script = (
        'import torch, numpy, numba, librosa, cv2, av, facexlib, gfpgan, safetensors; '
        'from src.utils.face_enhancer import enhancer_generator_with_len; '
        'from src.utils.preprocess import CropAndExtract; '
        'assert torch.cuda.is_available(), "GPU CUDA no detectada"; '
        'assert torch.__version__.startswith("1.12.1"), torch.__version__; '
        'print("ENV_OK", torch.__version__, torch.version.cuda)'
    )
    res = subprocess.run([str(PY_ST), '-c', script], cwd=REPO,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, errors='replace')
    if res.returncode == 0 and 'ENV_OK' in res.stdout:
        say(res.stdout.strip())
        return True
    say('Entorno Python por restaurar: ' + '\n'.join(res.stdout.splitlines()[-3:]))
    return False


def ensure_conda():
    if (MFG / 'bin/conda').is_file():
        return
    if MFG.exists():
        raise RuntimeError('Miniforge incompleto en /content/miniforge3: se conserva para revisión.')
    installer = Path('/content/FRAQX_Miniforge_installer.sh')
    say('Instalando Miniforge temporal para Python 3.8.')
    cmd(['curl', '--fail', '--location', '--retry', '4', '--retry-delay', '3',
         '--output', str(installer),
         'https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh'])
    cmd(['bash', str(installer), '-b', '-p', str(MFG)])
    installer.unlink(missing_ok=True)
    if not (MFG / 'bin/conda').is_file():
        raise RuntimeError('Miniforge no se instaló correctamente.')


def ensure_environment():
    if env_ready():
        say('Entorno Python 3.8/CUDA: reutilizado sin reinstalar')
        return
    if PY_ST.is_file():
        raise RuntimeError('Entorno Python existente pero no validado: se conserva intacto para auditoría.')
    if shutil.disk_usage('/content').free < 7_000_000_000:
        raise RuntimeError('Espacio insuficiente para restaurar Python/PyTorch en /content.')
    ensure_conda()
    os.environ['CONDA_PLUGINS_AUTO_ACCEPT_TOS'] = 'true'
    if not PY_ST.is_file():
        say('Creando entorno Python 3.8.20.')
        cmd([str(MFG / 'bin/conda'), 'create', '-y', '-p',
             str(PY_ST.parent.parent), 'python=3.8.20', 'pip=24.0',
             'setuptools=69.5.1', 'wheel=0.43.0'], env=os.environ.copy())
    if not PY_ST.is_file():
        raise RuntimeError('No se pudo crear el ejecutable Python del entorno.')
    say('Instalando/recuperando PyTorch CUDA 11.3 y dependencias fijadas.')
    cmd([str(PY_ST), '-m', 'pip', 'install', '--disable-pip-version-check',
         'torch==1.12.1+cu113', 'torchvision==0.13.1+cu113',
         'torchaudio==0.12.1', '--extra-index-url',
         'https://download.pytorch.org/whl/cu113'])
    cmd([str(PY_ST), '-m', 'pip', 'install', '--disable-pip-version-check'] + PY_REQUIREMENTS)
    if not env_ready():
        raise RuntimeError('El entorno fue instalado, pero no pasó la prueba CUDA/importaciones. Revisa el registro.')
    deps = BASE / 'app' / 'DEPENDENCIAS_FRAQX_v1_2_2.txt'
    deps_content = ('Python 3.8.20\nPyTorch 1.12.1+cu113\nTorchVision 0.13.1+cu113\nTorchaudio 0.12.1\n'
                    + '\n'.join(PY_REQUIREMENTS) + '\n')
    if deps.exists():
        if deps.read_text(encoding='utf-8') != deps_content:
            raise RuntimeError('Conflicto en manifiesto; no se sobrescribe: ' + str(deps))
    else:
        with deps.open('x', encoding='utf-8') as writer:
            writer.write(deps_content)
    say('Entorno Python 3.8/CUDA: PASS')


def ensure_gradio():
    # Comprobar en un subproceso evita dejar una versión incompatible
    # en sys.modules si hay que instalar o cambiar Gradio.
    result = subprocess.run(
        [sys.executable, '-c', 'import gradio; print(gradio.__version__)'],
        stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
    version = result.stdout.strip().splitlines()[-1] if result.returncode == 0 and result.stdout.strip() else ''
    try:
        compatible = 4 <= int(version.split('.')[0]) < 6
    except (ValueError, IndexError):
        compatible = False
    if not compatible:
        say('Instalando Gradio compatible para Colab')
        cmd([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
             'gradio>=4.44,<6'])
    import gradio
    if not (4 <= int(gradio.__version__.split('.')[0]) < 6):
        raise RuntimeError('La versión Gradio no es compatible con v1.2.1.')
    say('Gradio ' + gradio.__version__ + ': PASS')


# ===== BARRERA DE SEGURIDAD v1.2.2 =====
# Los controles son deliberadamente FALSE en la distribución.
# Una ejecución accidental de "Ejecutar todas" no recupera ni descarga nada.
AUTORIZO_RECUPERACION = False
AUTORIZO_DESCARGAS = False


def gpu_usable():
    # Prueba de SOLO lectura, no depende de CUDA/PyTorch ni de Drive.
    exe = shutil.which('nvidia-smi')
    if not exe:
        return False, 'nvidia-smi no encontrado'
    try:
        result = subprocess.run([exe, '-L'], capture_output=True, text=True, timeout=15)
    except (OSError, subprocess.TimeoutExpired) as error:
        return False, str(error)
    if result.returncode != 0 or not any(line.strip().startswith('GPU ') for line in result.stdout.splitlines()):
        return False, (result.stderr or result.stdout or 'ninguna GPU asignada').strip()[-400:]
    return True, result.stdout.strip()


def pending_downloads():
    # Todo lo siguiente es lectura local. No crea rutas ni conecta a Internet.
    required = []
    if not (REPO / 'inference.py').is_file():
        required.append('Repositorio SadTalker ausente')
    if not (MFG / 'bin/conda').is_file():
        required.append('Instalador Miniforge necesario')
    if not PY_ST.is_file():
        required.append('Dependencias Python/PyTorch pendientes')
    # Conservar idéntica política del notebook original: sin descarga de pesos SadTalker.
    for name, (url, minimum) in GFPGAN_MODELS.items():
        store = BASE / 'models' / 'gfpgan' / 'weights' / name
        candidates = [store, BASE / 'models' / name,
                      REPO / 'checkpoints' / name, REPO / 'gfpgan' / 'weights' / name]
        if not any(is_good(path, minimum) for path in candidates):
            required.append('Peso GFPGAN no localizado: ' + name)
    try:
        import importlib.metadata as metadata
        major = int(metadata.version('gradio').split('.')[0])
        if not 4 <= major < 6:
            required.append('Gradio compatible pendiente')
    except Exception:
        required.append('Gradio compatible pendiente')
    return required


def guard_preflight():
    ok, detail = gpu_usable()
    if not ok:
        print('[FRAQX] GPU_NO_DISPONIBLE: ' + detail)
        print('[FRAQX] PARADA_SEGURA: sin montaje, instalaciones, descargas ni escritura.')
        return False
    print('[FRAQX] GPU_NVIDIA_CONFIRMADA: ' + detail.splitlines()[0])
    if not BASE.is_dir():
        print('[FRAQX] DRIVE_NO_MONTADO_O_CARPETA_AUSENTE: ' + str(BASE))
        print('[FRAQX] Monta Drive mediante la celda independiente y confirma la carpeta.')
        return False
    for binary in ('git','curl','ffmpeg','ffprobe'):
        if not shutil.which(binary):
            print('[FRAQX] UTILIDAD_FALTANTE: ' + binary)
            return False
    # Checkpoint hashes no son conocidos: validar presencia y tamaño sin asumir integridad.
    for name, minimum in SADTALKER_MODELS.items():
        if not is_good(BASE / 'models' / name, minimum):
            print('[FRAQX] CHECKPOINT_INCOMPLETO_O_AUSENTE: ' + name)
            return False
    # Evitar acciones sobre repo/entorno incompleto existente.
    if REPO.exists() and not (REPO / 'inference.py').is_file():
        print('[FRAQX] REPO_PARCIAL: revisión manual requerida.')
        return False
    if MFG.exists() and not (MFG / 'bin/conda').is_file():
        print('[FRAQX] MINIFORGE_PARCIAL: revisión manual requerida.')
        return False
    if PY_ST.is_file():
        print('[FRAQX] AVISO: Python SadTalker existente. Se validará antes de cualquier cambio.')
    downloads = pending_downloads()
    print('[FRAQX] PLAN:')
    for item in downloads or ['No hay descargas evidentes pendientes en inspección estática']:
        print('   - ' + item)
    if downloads and not AUTORIZO_DESCARGAS:
        print('[FRAQX] BLOQUEO_DE_DESCARGAS: falta autorización específica (AUTORIZO_DESCARGAS=True).')
        return False
    if not AUTORIZO_RECUPERACION:
        print('[FRAQX] MODO_SEGURO: recuperación no autorizada. No se escribió ningún archivo.')
        return False
    return True


def main():
    started = time.monotonic()
    if not guard_preflight():
        return
    # Solo a partir de aquí hay acciones que pueden modificar /content y Drive.
    # Se exige GPU, Drive montado, archivos mínimos y autorización expresa.
    init_drive_logger()
    try:
        say('=== RECUPERACION CONTROLADA FRAQX v1.2.3 ===')
        ensure_gpu_and_utilities()
        ensure_repo()
        ensure_models()
        ensure_environment()
        ensure_gradio()
        say('RESULTADO: PASS_FRAQX_ENTORNO_LISTO')
        say('El entorno temporal quedó listo. No se ha iniciado Gradio ni se ha ejecutado SadTalker.')
    except Exception as error:
        say('FAIL_FRAQX_RECUPERACION: ' + str(error))
        if LOG:
            with LOG.open('a', encoding='utf-8') as out:
                traceback.print_exc(file=out)
        raise
    finally:
        say('Tiempo de preparación/lanzamiento: ' + str(round(time.monotonic()-started,1)) + ' s')


if __name__ == '__main__':
    main()


In [ ]:
# CELDA 6 — Abrir interfaz Gradio v1.2.3 (requiere GPU + recuperación confirmada)
# No genera ni instala nada al iniciar. Los botones generan con GPU y guardan por segmentos.
from pathlib import Path
import hashlib, runpy, shutil, subprocess
AUTORIZO_ABRIR_INTERFAZ = False
AUTORIZO_ENLACE_PUBLICO = False
BASE = Path('/content/drive/MyDrive/FRAQX_AI_Avatar_Studio')
ENGINE = BASE/'app'/'fraqx_render_recuperable_v123.py'
APP = BASE/'app'/'FRAQX_AI_Avatar_Studio_v1_2_3.py'
ESPERADO_ENGINE = '8bdd4a7a64600838aeb3ab602a441a58ee5f0abcd8939eec8dd31206eee686e7'
ESPERADO_APP = '414a7dd1d8716a025ebc31ec9815ca7bc528cae0876dd419062e2e110ccf29e9'

def _sha(path):
    h=hashlib.sha256()
    with path.open('rb') as inp:
        for block in iter(lambda: inp.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

print('=== FRAQX v1.2.3: APERTURA CONTROLADA ===')
if not AUTORIZO_ABRIR_INTERFAZ:
    print('PARADA_SEGURA: interfaz no autorizada (AUTORIZO_ABRIR_INTERFAZ=False).')
elif not BASE.is_dir() or any(not p.is_file() for p in (ENGINE,APP)):
    print('PARADA_SEGURA: Drive o motor v1.2.3 no disponibles; ejecuta inventario/publicación.')
elif _sha(ENGINE) != ESPERADO_ENGINE or _sha(APP) != ESPERADO_APP:
    print('PARADA_SEGURA: módulo de aplicación diferente al validado.')
elif not shutil.which('nvidia-smi') or subprocess.run(['nvidia-smi','-L'],capture_output=True).returncode != 0:
    print('PARADA_SEGURA: GPU NVIDIA no asignada.')
elif not Path('/content/SadTalker/inference.py').is_file() or not Path('/content/miniforge3/envs/sadtalker/bin/python').is_file():
    print('PARADA_SEGURA: entorno SadTalker aún no recuperado.')
else:
    # El directorio app debe estar en sys.path para importar el motor del mismo nombre.
    import sys, os
    if str(APP.parent) not in sys.path:
        sys.path.insert(0, str(APP.parent))
    os.environ['FRAQX_SHARE_GRADIO']='1' if AUTORIZO_ENLACE_PUBLICO else '0'
    print('LANZANDO GRADIO v1.2.3 (sin iniciar ningún render).')
    runpy.run_path(str(APP),run_name='__main__')
